In [2]:
"""
Google News -> non-paywalled article scraper for multiple 2025 NFL teams (Selenium)

Middle-ground fast mode:
- Faster than the original version
- Still keeps moderate throttling to reduce immediate blocking risk
- Uses conditional waits for article page load instead of long fixed sleeps

Goal:
- For each configured team's 2025 regular-season games:
  - Collect up to TARGET_PER_GAME NON-PAYWALLED articles per game
  - Each output row: team, source, author, title, text_body (+ helpful fields)
- Paywalled articles are skipped and DO NOT count toward the target
- The script paginates Google News results until it reaches the target
  or runs out of results

Dependencies:
  pip install selenium beautifulsoup4

Notes:
- Run HEADLESS=False first in case Google shows consent/CAPTCHA
- Author extraction is heuristic
- Body extraction is heuristic and some JS-heavy sites may still fail
"""

from __future__ import annotations

import csv
import os
import random
import re
import time
from typing import Dict, List, Optional, Tuple
from urllib.parse import quote_plus, urlparse

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
    NoSuchElementException,
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ----------------------------
# CONFIG
# ----------------------------

TARGET_PER_GAME = 20
HEADLESS = False

PAGELOAD_TIMEOUT_S = 15
WAIT_S = 8
ARTICLE_WAIT_TIMEOUT_S = 5

# Middle-ground throttling
SLEEP_BETWEEN_ARTICLES = (0.6, 1.5)
SLEEP_BETWEEN_PAGES = (0.5, 1.2)
SLEEP_BETWEEN_GAMES = (2.0, 5.0)
POST_ARTICLE_LOAD_SETTLE = (0.2, 0.6)
POST_PAGE_CLICK_SETTLE = (0.7, 1.5)
BACKOFF_BASE = 1.5

GOOGLE_NEWS_NUM_PER_PAGE = 20

MAX_RESULTS_PAGES_PER_GAME = 10
MAX_ARTICLE_VISITS_PER_GAME = 80

OUT_DIR = "google_news_outputs"

# Leave empty to run all teams
SELECTED_TEAMS = [
    # "Philadelphia Eagles",
    # "Dallas Cowboys",
    # "New England Patriots",
    # "Cincinnati Bengals",
    # "Chicago Bears",
    # "Tampa Bay Buccaneers",
    # "Seattle Seahawks",
    # "Buffalo Bills",
    # "Indianapolis Colts",
    # "Kansas City Chiefs",
]


# ----------------------------
# 2025 REGULAR-SEASON SCHEDULES
# ----------------------------

TEAM_SCHEDULES_2025: Dict[str, List[Dict[str, str]]] = {
    "Philadelphia Eagles": [
        {"game_no": "1",  "date": "2025-09-04", "opponent": "Dallas Cowboys",         "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Kansas City Chiefs",     "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Los Angeles Rams",       "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Tampa Bay Buccaneers",   "home_away": "away"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Denver Broncos",         "home_away": "home"},
        {"game_no": "6",  "date": "2025-10-09", "opponent": "New York Giants",        "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-19", "opponent": "Minnesota Vikings",      "home_away": "away"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "New York Giants",        "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-10", "opponent": "Green Bay Packers",      "home_away": "away"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Detroit Lions",          "home_away": "home"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Dallas Cowboys",         "home_away": "away"},
        {"game_no": "12", "date": "2025-11-28", "opponent": "Chicago Bears",          "home_away": "home"},
        {"game_no": "13", "date": "2025-12-08", "opponent": "Los Angeles Chargers",   "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Las Vegas Raiders",      "home_away": "home"},
        {"game_no": "15", "date": "2025-12-20", "opponent": "Washington Commanders",  "home_away": "away"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Buffalo Bills",          "home_away": "away"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Washington Commanders",  "home_away": "home"},
    ],
    "Dallas Cowboys": [
        {"game_no": "1",  "date": "2025-09-04", "opponent": "Philadelphia Eagles",    "home_away": "away"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "New York Giants",        "home_away": "home"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Chicago Bears",          "home_away": "away"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Green Bay Packers",      "home_away": "home"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "New York Jets",          "home_away": "away"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "Carolina Panthers",      "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-19", "opponent": "Washington Commanders",  "home_away": "home"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "Denver Broncos",         "home_away": "away"},
        {"game_no": "9",  "date": "2025-11-03", "opponent": "Arizona Cardinals",      "home_away": "home"},
        {"game_no": "10", "date": "2025-11-17", "opponent": "Las Vegas Raiders",      "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Philadelphia Eagles",    "home_away": "home"},
        {"game_no": "12", "date": "2025-11-27", "opponent": "Kansas City Chiefs",     "home_away": "home"},
        {"game_no": "13", "date": "2025-12-04", "opponent": "Detroit Lions",          "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Minnesota Vikings",      "home_away": "home"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Los Angeles Chargers",   "home_away": "home"},
        {"game_no": "16", "date": "2025-12-25", "opponent": "Washington Commanders",  "home_away": "away"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "New York Giants",        "home_away": "away"},
    ],
    "New England Patriots": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "Las Vegas Raiders",      "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Miami Dolphins",         "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Pittsburgh Steelers",    "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Carolina Panthers",      "home_away": "home"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Buffalo Bills",          "home_away": "away"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "New Orleans Saints",     "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-19", "opponent": "Tennessee Titans",       "home_away": "away"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "Cleveland Browns",       "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-02", "opponent": "Atlanta Falcons",        "home_away": "home"},
        {"game_no": "10", "date": "2025-11-09", "opponent": "Tampa Bay Buccaneers",   "home_away": "away"},
        {"game_no": "11", "date": "2025-11-13", "opponent": "New York Jets",          "home_away": "home"},
        {"game_no": "12", "date": "2025-11-23", "opponent": "Cincinnati Bengals",     "home_away": "away"},
        {"game_no": "13", "date": "2025-12-01", "opponent": "New York Giants",        "home_away": "home"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Buffalo Bills",          "home_away": "home"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Baltimore Ravens",       "home_away": "away"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "New York Jets",          "home_away": "away"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Miami Dolphins",         "home_away": "home"},
    ],
    "Cincinnati Bengals": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "Cleveland Browns",       "home_away": "away"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Jacksonville Jaguars",   "home_away": "home"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Minnesota Vikings",      "home_away": "away"},
        {"game_no": "4",  "date": "2025-09-29", "opponent": "Denver Broncos",         "home_away": "away"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Detroit Lions",          "home_away": "home"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "Green Bay Packers",      "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-16", "opponent": "Pittsburgh Steelers",    "home_away": "home"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "New York Jets",          "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-02", "opponent": "Chicago Bears",          "home_away": "home"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Pittsburgh Steelers",    "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "New England Patriots",   "home_away": "home"},
        {"game_no": "12", "date": "2025-11-27", "opponent": "Baltimore Ravens",       "home_away": "away"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Buffalo Bills",          "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Baltimore Ravens",       "home_away": "home"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Miami Dolphins",         "home_away": "away"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Arizona Cardinals",      "home_away": "home"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Cleveland Browns",       "home_away": "home"},
    ],
    "Chicago Bears": [
        {"game_no": "1",  "date": "2025-09-08", "opponent": "Minnesota Vikings",      "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Detroit Lions",          "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Dallas Cowboys",         "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Las Vegas Raiders",      "home_away": "away"},
        {"game_no": "5",  "date": "2025-10-13", "opponent": "Washington Commanders",  "home_away": "away"},
        {"game_no": "6",  "date": "2025-10-19", "opponent": "New Orleans Saints",     "home_away": "home"},
        {"game_no": "7",  "date": "2025-10-26", "opponent": "Baltimore Ravens",       "home_away": "away"},
        {"game_no": "8",  "date": "2025-11-02", "opponent": "Cincinnati Bengals",     "home_away": "away"},
        {"game_no": "9",  "date": "2025-11-09", "opponent": "New York Giants",        "home_away": "home"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Minnesota Vikings",      "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Pittsburgh Steelers",    "home_away": "home"},
        {"game_no": "12", "date": "2025-11-28", "opponent": "Philadelphia Eagles",    "home_away": "away"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Green Bay Packers",      "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Cleveland Browns",       "home_away": "home"},
        {"game_no": "15", "date": "2025-12-20", "opponent": "Green Bay Packers",      "home_away": "home"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "San Francisco 49ers",    "home_away": "away"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Detroit Lions",          "home_away": "home"},
    ],
    "Tampa Bay Buccaneers": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "Atlanta Falcons",        "home_away": "away"},
        {"game_no": "2",  "date": "2025-09-15", "opponent": "Houston Texans",         "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "New York Jets",          "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Philadelphia Eagles",    "home_away": "home"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Seattle Seahawks",       "home_away": "away"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "San Francisco 49ers",    "home_away": "home"},
        {"game_no": "7",  "date": "2025-10-20", "opponent": "Detroit Lions",          "home_away": "away"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "New Orleans Saints",     "home_away": "away"},
        {"game_no": "9",  "date": "2025-11-09", "opponent": "New England Patriots",   "home_away": "home"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Buffalo Bills",          "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Los Angeles Rams",       "home_away": "away"},
        {"game_no": "12", "date": "2025-11-30", "opponent": "Arizona Cardinals",      "home_away": "home"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "New Orleans Saints",     "home_away": "home"},
        {"game_no": "14", "date": "2025-12-11", "opponent": "Atlanta Falcons",        "home_away": "home"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Carolina Panthers",      "home_away": "away"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Miami Dolphins",         "home_away": "away"},
        {"game_no": "17", "date": "2026-01-03", "opponent": "Carolina Panthers",      "home_away": "home"},
    ],
    "Seattle Seahawks": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "San Francisco 49ers",    "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Pittsburgh Steelers",    "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "New Orleans Saints",     "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-25", "opponent": "Arizona Cardinals",      "home_away": "away"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Tampa Bay Buccaneers",   "home_away": "home"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "Jacksonville Jaguars",   "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-20", "opponent": "Houston Texans",         "home_away": "home"},
        {"game_no": "8",  "date": "2025-11-02", "opponent": "Washington Commanders",  "home_away": "away"},
        {"game_no": "9",  "date": "2025-11-09", "opponent": "Arizona Cardinals",      "home_away": "home"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Los Angeles Rams",       "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Tennessee Titans",       "home_away": "away"},
        {"game_no": "12", "date": "2025-11-30", "opponent": "Minnesota Vikings",      "home_away": "home"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Atlanta Falcons",        "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Indianapolis Colts",     "home_away": "home"},
        {"game_no": "15", "date": "2025-12-18", "opponent": "Los Angeles Rams",       "home_away": "home"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Carolina Panthers",      "home_away": "away"},
        {"game_no": "17", "date": "2026-01-03", "opponent": "San Francisco 49ers",    "home_away": "away"},
    ],
    "Buffalo Bills": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "Baltimore Ravens",       "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "New York Jets",          "home_away": "away"},
        {"game_no": "3",  "date": "2025-09-18", "opponent": "Miami Dolphins",         "home_away": "home"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "New Orleans Saints",     "home_away": "home"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "New England Patriots",   "home_away": "home"},
        {"game_no": "6",  "date": "2025-10-13", "opponent": "Atlanta Falcons",        "home_away": "away"},
        {"game_no": "7",  "date": "2025-10-26", "opponent": "Carolina Panthers",      "home_away": "away"},
        {"game_no": "8",  "date": "2025-11-02", "opponent": "Kansas City Chiefs",     "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-09", "opponent": "Miami Dolphins",         "home_away": "away"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Tampa Bay Buccaneers",   "home_away": "home"},
        {"game_no": "11", "date": "2025-11-20", "opponent": "Houston Texans",         "home_away": "away"},
        {"game_no": "12", "date": "2025-11-30", "opponent": "Pittsburgh Steelers",    "home_away": "away"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Cincinnati Bengals",     "home_away": "home"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "New England Patriots",   "home_away": "away"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Cleveland Browns",       "home_away": "away"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Philadelphia Eagles",    "home_away": "home"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "New York Jets",          "home_away": "home"},
    ],
    "Indianapolis Colts": [
        {"game_no": "1",  "date": "2025-09-07", "opponent": "Miami Dolphins",         "home_away": "home"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Denver Broncos",         "home_away": "home"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "Tennessee Titans",       "home_away": "away"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Los Angeles Rams",       "home_away": "away"},
        {"game_no": "5",  "date": "2025-10-05", "opponent": "Las Vegas Raiders",      "home_away": "home"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "Arizona Cardinals",      "home_away": "home"},
        {"game_no": "7",  "date": "2025-10-19", "opponent": "Los Angeles Chargers",   "home_away": "away"},
        {"game_no": "8",  "date": "2025-10-26", "opponent": "Tennessee Titans",       "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-02", "opponent": "Pittsburgh Steelers",    "home_away": "away"},
        {"game_no": "10", "date": "2025-11-09", "opponent": "Atlanta Falcons",        "home_away": "home"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Kansas City Chiefs",     "home_away": "away"},
        {"game_no": "12", "date": "2025-11-30", "opponent": "Houston Texans",         "home_away": "home"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Jacksonville Jaguars",   "home_away": "away"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Seattle Seahawks",       "home_away": "away"},
        {"game_no": "15", "date": "2025-12-22", "opponent": "San Francisco 49ers",    "home_away": "home"},
        {"game_no": "16", "date": "2025-12-28", "opponent": "Jacksonville Jaguars",   "home_away": "home"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Houston Texans",         "home_away": "away"},
    ],
    "Kansas City Chiefs": [
        {"game_no": "1",  "date": "2025-09-05", "opponent": "Los Angeles Chargers",   "home_away": "away"},
        {"game_no": "2",  "date": "2025-09-14", "opponent": "Philadelphia Eagles",    "home_away": "home"},
        {"game_no": "3",  "date": "2025-09-21", "opponent": "New York Giants",        "home_away": "away"},
        {"game_no": "4",  "date": "2025-09-28", "opponent": "Baltimore Ravens",       "home_away": "home"},
        {"game_no": "5",  "date": "2025-10-06", "opponent": "Jacksonville Jaguars",   "home_away": "away"},
        {"game_no": "6",  "date": "2025-10-12", "opponent": "Detroit Lions",          "home_away": "home"},
        {"game_no": "7",  "date": "2025-10-19", "opponent": "Las Vegas Raiders",      "home_away": "home"},
        {"game_no": "8",  "date": "2025-10-27", "opponent": "Washington Commanders",  "home_away": "home"},
        {"game_no": "9",  "date": "2025-11-02", "opponent": "Buffalo Bills",          "home_away": "away"},
        {"game_no": "10", "date": "2025-11-16", "opponent": "Denver Broncos",         "home_away": "away"},
        {"game_no": "11", "date": "2025-11-23", "opponent": "Indianapolis Colts",     "home_away": "home"},
        {"game_no": "12", "date": "2025-11-27", "opponent": "Dallas Cowboys",         "home_away": "away"},
        {"game_no": "13", "date": "2025-12-07", "opponent": "Houston Texans",         "home_away": "home"},
        {"game_no": "14", "date": "2025-12-14", "opponent": "Los Angeles Chargers",   "home_away": "home"},
        {"game_no": "15", "date": "2025-12-21", "opponent": "Tennessee Titans",       "home_away": "away"},
        {"game_no": "16", "date": "2025-12-25", "opponent": "Denver Broncos",         "home_away": "home"},
        {"game_no": "17", "date": "2026-01-04", "opponent": "Las Vegas Raiders",      "home_away": "away"},
    ],
}


# ----------------------------
# PAYWALL DETECTION
# ----------------------------

PAYWALL_KEYWORDS = [
    "subscribe to continue",
    "subscribe now",
    "subscription required",
    "subscriber-only",
    "sign in to continue",
    "register to continue",
    "create an account",
    "already a subscriber",
    "to keep reading",
    "unlimited access",
    "unlock this article",
    "this content is available to subscribers",
    "this article is for subscribers",
    "become a subscriber",
    "membership required",
    "you've reached your limit",
    "free articles remaining",
    "metered",
    "turn off your ad blocker",
    "tinypass",
    "piano",
    "paywall",
    "zephr",
    "subscribe with google",
]

PAYWALL_CSS_SELECTORS = [
    '[id*="paywall" i]',
    '[class*="paywall" i]',
    '[id*="subscribe" i]',
    '[class*="subscribe" i]',
    '[id*="subscription" i]',
    '[class*="subscription" i]',
    '[class*="meter" i]',
    '[id*="meter" i]',
    '[class*="tp-" i]',
    '[id*="tp-" i]',
    '[class*="piano" i]',
    '[id*="piano" i]',
    '[class*="zephr" i]',
    '[id*="zephr" i]',
    '[class*="regwall" i]',
    '[id*="regwall" i]',
    '[class*="overlay" i]',
    '[class*="modal" i]',
]

VENDOR_MARKERS = [
    "tinypass",
    "piano.io",
    "sandbox.piano",
    "zephr",
    "arcxp",
    "gateway",
]


# ----------------------------
# UTIL
# ----------------------------

def jitter_sleep(a_b: Tuple[float, float]) -> None:
    time.sleep(random.uniform(*a_b))


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()


def canonicalize_url(url: str) -> str:
    try:
        u = urlparse(url)
        return f"{u.scheme}://{u.netloc}{u.path}"
    except Exception:
        return url


def slugify_team_name(team_name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", team_name.lower()).strip("_")


def safe_get(driver: webdriver.Chrome, url: str, retries: int = 3) -> None:
    for attempt in range(1, retries + 1):
        try:
            driver.set_page_load_timeout(PAGELOAD_TIMEOUT_S)
            driver.get(url)
            return
        except (TimeoutException, WebDriverException):
            if attempt == retries:
                raise
            time.sleep(BACKOFF_BASE * attempt + random.uniform(0.0, 1.0))


def likely_google_consent_or_block(html: str) -> bool:
    s = (html or "").lower()
    return (
        ("before you continue to google" in s)
        or ("consent.google.com" in s)
        or ("unusual traffic" in s)
        or ("our systems have detected unusual traffic" in s)
        or ("sorry" in s and "robots" in s)
    )


def build_google_news_search_url(query: str, num: int = 20) -> str:
    q = quote_plus(query)
    return f"https://www.google.com/search?q={q}&tbm=nws&num={num}&hl=en&gl=us"


def build_game_query(team_name: str, game: Dict[str, str]) -> str:
    game_date = game["date"]
    opp = game["opponent"]
    ha = "at" if game["home_away"] == "away" else "vs"
    return f'{team_name} {ha} "{opp}" {game_date} recap analysis'


# ----------------------------
# PAYWALL HELPERS
# ----------------------------

def looks_like_paywall_html(html: str) -> Tuple[bool, str]:
    low = (html or "").lower()

    hits = [kw for kw in PAYWALL_KEYWORDS if kw in low]
    if hits:
        return True, f"keyword:{hits[0]}"

    for marker in VENDOR_MARKERS:
        if marker in low:
            return True, f"vendor_marker:{marker}"

    return False, ""


def looks_like_paywall_dom(driver: webdriver.Chrome) -> Tuple[bool, str]:
    try:
        try:
            body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
        except Exception:
            body_text = ""

        for kw in PAYWALL_KEYWORDS:
            if kw in body_text:
                return True, f"body_text:{kw}"

        for sel in PAYWALL_CSS_SELECTORS:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if not els:
                continue

            for el in els:
                try:
                    if not el.is_displayed():
                        continue

                    if ("modal" in sel or "overlay" in sel):
                        t = (el.text or "").lower()
                        if any(k in t for k in ["subscribe", "subscriber", "sign in", "register", "limit"]):
                            return True, f"selector:{sel}"
                        continue

                    return True, f"selector:{sel}"
                except Exception:
                    continue

        return False, ""
    except Exception:
        return False, ""


def is_probably_paywalled(driver: webdriver.Chrome, html: str) -> Tuple[bool, str]:
    pw1, r1 = looks_like_paywall_html(html)
    if pw1:
        return True, r1

    pw2, r2 = looks_like_paywall_dom(driver)
    if pw2:
        return True, r2

    return False, ""


# ----------------------------
# EXTRACTION
# ----------------------------

def extract_author_from_meta(soup: BeautifulSoup) -> Optional[str]:
    selectors = [
        'meta[name="author"]',
        'meta[property="article:author"]',
        'meta[name="parsely-author"]',
    ]
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag:
            c = clean_text(tag.get("content", ""))
            if c:
                return re.sub(r"^\s*by\s+", "", c, flags=re.I).strip()
    return None


def extract_title_from_article(soup: BeautifulSoup) -> Optional[str]:
    og = soup.select_one('meta[property="og:title"]')
    if og and clean_text(og.get("content", "")):
        return clean_text(og.get("content", ""))

    if soup.title and soup.title.string:
        return clean_text(soup.title.string)

    h1 = soup.find("h1")
    if h1:
        return clean_text(h1.get_text(" ", strip=True))

    return None


def extract_main_text_from_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "canvas", "header", "footer", "aside"]):
        tag.decompose()

    article = soup.find("article")
    if article:
        txt = clean_text(article.get_text(" ", strip=True))
        if len(txt) >= 400:
            return txt

    main = soup.find("main")
    if main:
        txt = clean_text(main.get_text(" ", strip=True))
        if len(txt) >= 400:
            return txt

    best_txt = ""
    best_score = 0

    for node in soup.find_all(["div", "section"]):
        txt = clean_text(node.get_text(" ", strip=True))
        if len(txt) < 400:
            continue

        punct = len(re.findall(r"[.!?]", txt))
        score = len(txt) + 50 * min(punct, 30)

        class_id = " ".join([
            " ".join(node.get("class", []) or []),
            node.get("id", "") or ""
        ]).lower()

        if any(k in class_id for k in ["nav", "menu", "footer", "header", "sidebar", "subscribe", "promo"]):
            score *= 0.6

        if score > best_score:
            best_score = score
            best_txt = txt

    return best_txt


# ----------------------------
# GOOGLE NEWS RESULT PARSING
# ----------------------------

def collect_result_cards(driver: webdriver.Chrome) -> List[BeautifulSoup]:
    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select("div.SoaBEf")
    if cards:
        return cards

    anchors = soup.select("a.WlydOe")
    out = []
    for a in anchors:
        parent = a.find_parent("g-card") or a.find_parent("div")
        if parent:
            out.append(parent)
    return out


def parse_card(card: BeautifulSoup) -> Dict[str, Optional[str]]:
    a = card.select_one("a.WlydOe")
    title = clean_text(a.get_text(" ", strip=True)) if a else None
    link = a.get("href") if a else None

    if link and link.startswith("/"):
        link = "https://www.google.com" + link

    source = None
    src = card.select_one(".CEMjEf, .MgUUmf span, span.xQ82C")
    if src:
        source = clean_text(src.get_text(" ", strip=True))

    return {
        "title": title,
        "link": link,
        "source": source,
    }


# ----------------------------
# DRIVER
# ----------------------------

def start_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")

    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)

    opts.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)

    try:
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"}
        )
    except Exception:
        pass

    return driver


# ----------------------------
# TAB / PAGE HELPERS
# ----------------------------

def open_in_new_tab(driver: webdriver.Chrome, url: str) -> None:
    driver.execute_script("window.open(arguments[0], '_blank');", url)
    driver.switch_to.window(driver.window_handles[-1])


def close_current_tab_and_return(driver: webdriver.Chrome) -> None:
    driver.close()
    driver.switch_to.window(driver.window_handles[0])


def click_next_results_page(driver: webdriver.Chrome) -> bool:
    try:
        nxt = driver.find_element(By.CSS_SELECTOR, "a#pnnext")
        jitter_sleep((0.3, 0.8))
        nxt.click()
        return True
    except NoSuchElementException:
        return False
    except Exception:
        return False


def ensure_results_loaded(driver: webdriver.Chrome) -> None:
    WebDriverWait(driver, WAIT_S).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.WlydOe, div.SoaBEf"))
    )


def wait_for_article_loaded(driver: webdriver.Chrome, timeout: int = ARTICLE_WAIT_TIMEOUT_S) -> None:
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


# ----------------------------
# GAME SCRAPER
# ----------------------------

def scrape_non_paywalled_for_game(
    driver: webdriver.Chrome,
    team_name: str,
    game: Dict[str, str],
    target: int,
) -> List[Dict[str, str]]:
    query = build_game_query(team_name, game)
    search_url = build_google_news_search_url(query, num=GOOGLE_NEWS_NUM_PER_PAGE)

    safe_get(driver, search_url)

    if likely_google_consent_or_block(driver.page_source):
        print("[!] Google consent/block page detected.")
        print("    Complete consent/CAPTCHA in the opened browser window, then re-run.")
        time.sleep(10)

    ensure_results_loaded(driver)

    seen_links = set()
    rows: List[Dict[str, str]] = []
    pages_used = 0
    visits = 0

    while len(rows) < target:
        if pages_used >= MAX_RESULTS_PAGES_PER_GAME:
            break
        if visits >= MAX_ARTICLE_VISITS_PER_GAME:
            break

        cards = collect_result_cards(driver)

        new_refs: List[Dict[str, str]] = []
        for c in cards:
            d = parse_card(c)
            if not d.get("link") or not d.get("title"):
                continue

            can = canonicalize_url(d["link"])
            if can in seen_links:
                continue

            seen_links.add(can)
            new_refs.append({
                "source": d.get("source") or "",
                "title_hint": d.get("title") or "",
                "link": d["link"],
            })

        for ref in new_refs:
            if len(rows) >= target:
                break
            if visits >= MAX_ARTICLE_VISITS_PER_GAME:
                break

            visits += 1
            jitter_sleep(SLEEP_BETWEEN_ARTICLES)

            link = ref["link"]
            try:
                open_in_new_tab(driver, link)

                try:
                    wait_for_article_loaded(driver, timeout=ARTICLE_WAIT_TIMEOUT_S)
                except TimeoutException:
                    pass

                jitter_sleep(POST_ARTICLE_LOAD_SETTLE)

                html = driver.page_source

                paywalled, _reason = is_probably_paywalled(driver, html)
                if paywalled:
                    continue

                soup = BeautifulSoup(html, "html.parser")
                author = extract_author_from_meta(soup) or ""
                title = extract_title_from_article(soup) or ref.get("title_hint", "")
                body = extract_main_text_from_html(html) or ""

                low = html.lower()
                if len(body) < 250 and any(k in low for k in ["subscribe", "subscriber", "sign in", "register", "limit"]):
                    continue

                if len(body) < 250:
                    continue

                rows.append({
                    "team": team_name,
                    "game_no": game["game_no"],
                    "game_date": game["date"],
                    "opponent": game["opponent"],
                    "home_away": game["home_away"],
                    "query": query,
                    "source": ref.get("source", ""),
                    "author": author,
                    "title": title,
                    "text_body": body,
                    "url": link,
                })

            except Exception as e:
                print(f"[!] Article error | {team_name} game {game['game_no']}: {e}")
            finally:
                if len(driver.window_handles) > 1:
                    try:
                        close_current_tab_and_return(driver)
                    except Exception:
                        driver.switch_to.window(driver.window_handles[0])

        if len(rows) < target:
            pages_used += 1
            jitter_sleep(SLEEP_BETWEEN_PAGES)

            moved = click_next_results_page(driver)
            if not moved:
                break

            jitter_sleep(POST_PAGE_CLICK_SETTLE)

            if likely_google_consent_or_block(driver.page_source):
                print("[!] Hit Google consent/block mid-pagination; stopping this game early.")
                break

            try:
                ensure_results_loaded(driver)
            except TimeoutException:
                break

    return rows


# ----------------------------
# TEAM RUNNER
# ----------------------------

def run_team_games(
    team_name: str,
    games: List[Dict[str, str]],
    target_per_game: int = TARGET_PER_GAME,
    out_csv: Optional[str] = None,
) -> List[Dict[str, str]]:
    if len(games) != 17:
        print(f"[!] {team_name}: provided {len(games)} games; expected 17.")

    os.makedirs(OUT_DIR, exist_ok=True)

    if out_csv is None:
        out_csv = os.path.join(
            OUT_DIR,
            f"{slugify_team_name(team_name)}_2025_articles_non_paywalled.csv"
        )

    driver = start_driver()
    all_rows: List[Dict[str, str]] = []

    fieldnames = [
        "team",
        "game_no",
        "game_date",
        "opponent",
        "home_away",
        "query",
        "source",
        "author",
        "title",
        "text_body",
        "url",
    ]

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        try:
            for i, game in enumerate(games, start=1):
                print(
                    f"\n=== {team_name} | Game {i}/{len(games)} "
                    f"| {game['date']} | {game['home_away']} vs {game['opponent']} ==="
                )

                try:
                    rows = scrape_non_paywalled_for_game(
                        driver=driver,
                        team_name=team_name,
                        game=game,
                        target=target_per_game,
                    )
                except TimeoutException:
                    print("[!] Timeout on results page; backing off and continuing.")
                    time.sleep(BACKOFF_BASE + random.uniform(0.0, 2.0))
                    rows = []
                except WebDriverException as e:
                    print(f"[!] WebDriver error: {e}. Backing off and continuing.")
                    time.sleep(BACKOFF_BASE + random.uniform(0.0, 3.0))
                    rows = []

                print(
                    f"Collected {len(rows)} non-paywalled articles "
                    f"for {team_name} game {game['game_no']}."
                )

                for r in rows:
                    writer.writerow(r)

                f.flush()
                all_rows.extend(rows)

                jitter_sleep(SLEEP_BETWEEN_GAMES)

        finally:
            try:
                driver.quit()
            except Exception:
                pass

    print(f"\nDone. Wrote {len(all_rows)} rows to {out_csv}.")
    return all_rows


# ----------------------------
# MULTI-TEAM RUNNER
# ----------------------------

def run_selected_teams(
    selected_teams: Optional[List[str]] = None,
    target_per_game: int = TARGET_PER_GAME,
) -> Dict[str, List[Dict[str, str]]]:
    if not selected_teams:
        selected_teams = list(TEAM_SCHEDULES_2025.keys())

    output: Dict[str, List[Dict[str, str]]] = {}

    for team_name in selected_teams:
        if team_name not in TEAM_SCHEDULES_2025:
            print(f"[!] Skipping unknown team: {team_name}")
            continue

        print(f"\n{'#' * 80}")
        print(f"RUNNING TEAM: {team_name}")
        print(f"{'#' * 80}")

        rows = run_team_games(
            team_name=team_name,
            games=TEAM_SCHEDULES_2025[team_name],
            target_per_game=target_per_game,
        )
        output[team_name] = rows

    return output


# ----------------------------
# RUN
# ----------------------------

if __name__ == "__main__":
    all_results = run_selected_teams(
        selected_teams=SELECTED_TEAMS,
        target_per_game=TARGET_PER_GAME,
    )


################################################################################
RUNNING TEAM: Philadelphia Eagles
################################################################################

=== Philadelphia Eagles | Game 1/17 | 2025-09-04 | home vs Dallas Cowboys ===
Collected 4 non-paywalled articles for Philadelphia Eagles game 1.

=== Philadelphia Eagles | Game 2/17 | 2025-09-14 | away vs Kansas City Chiefs ===
Collected 3 non-paywalled articles for Philadelphia Eagles game 2.

=== Philadelphia Eagles | Game 3/17 | 2025-09-21 | home vs Los Angeles Rams ===
Collected 6 non-paywalled articles for Philadelphia Eagles game 3.

=== Philadelphia Eagles | Game 4/17 | 2025-09-28 | away vs Tampa Bay Buccaneers ===
Collected 4 non-paywalled articles for Philadelphia Eagles game 4.

=== Philadelphia Eagles | Game 5/17 | 2025-10-05 | home vs Denver Broncos ===
Collected 5 non-paywalled articles for Philadelphia Eagles game 5.

=== Philadelphia Eagles | Game 6/17 | 2025-10-09 | away vs N